# 3_7 — Publication figures & headline table (P2)

Built on cross-validation artifacts (3_4/3_5); no torch needed. All figures are in English with **object-cluster bootstrap 95% CIs** (not naive fold-CI). Saved to `results/analysis_figs/`.
Figures: (1) reliability diagram, (2) forest plots (AUPRC, post-prefix RMSE) with cluster CIs, (3) onset-vs-trajectory Pareto trade-off, (4) headline table.

In [1]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
TABLES=os.path.join(REPO,'results','analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB=os.path.join(REPO,'results','tables'); os.makedirs(PUB, exist_ok=True)
FIGS=os.path.join(REPO,'results','analysis_figs'); os.makedirs(FIGS, exist_ok=True)

In [2]:
from liquefaction_ai.evaluation import reliability_diagram, forest_plot, pareto_plot, headline_table
from IPython.display import display
REF = 'DPI-Flow'
samples = pd.read_csv(os.path.join(TABLES,'cv_grouped_samples.csv'))
summary = pd.read_csv(os.path.join(TABLES,'cv_grouped_summary.csv'))
clpath = os.path.join(TABLES,'significance_grouped_cluster_bootstrap.csv')
cluster = pd.read_csv(clpath) if os.path.exists(clpath) else None

## 1. Calibration — reliability diagram

In [3]:
fig, ece = reliability_diagram(samples, ref=REF, bins=10, out_dir=FIGS, suffix='grouped')
print('ECE =', round(ece, 4)); display(fig)

ECE = 0.0498


/Users/nikita/Desktop/projects/liquefaction-ai/src/liquefaction_ai/evaluation/publication.py:78: UserWarning: Glyph 8733 (\N{PROPORTIONAL TO}) missing from font(s) Lucida Grande.
  fig.tight_layout()
/Users/nikita/Desktop/projects/liquefaction-ai/src/liquefaction_ai/evaluation/publication.py:82: UserWarning: Glyph 8733 (\N{PROPORTIONAL TO}) missing from font(s) Lucida Grande.
  fig.savefig(os.path.join(out_dir, f"reliability_{ref}_{suffix}.{ext}"), dpi=150)


<Figure size 460x440 with 1 Axes>

## 2. Model comparison with cluster-bootstrap CIs (forest plots)

In [4]:
if cluster is not None:
    display(forest_plot(cluster, metric='AUPRC', higher_better=True, ref=REF, out_dir=FIGS))
    if 'traj_rmse_continuation' in cluster.columns:
        display(forest_plot(cluster, metric='traj_rmse_continuation', higher_better=False, ref=REF,
                            title='Post-prefix trajectory RMSE', out_dir=FIGS))
else:
    print('run 3_5 first to produce significance_grouped_cluster_bootstrap.csv')

<Figure size 640x590 with 1 Axes>

<Figure size 640x540 with 1 Axes>

## 3. Onset vs trajectory trade-off (Pareto balance)

In [5]:
display(pareto_plot(summary, x='Traj_RMSE_continuation', y='AUPRC', ref=REF, out_dir=FIGS))

<Figure size 620x500 with 1 Axes>

## 4. Publication headline table (CI from cluster bootstrap)

In [6]:
hl = headline_table(summary, cluster_df=cluster)
hl.to_csv(os.path.join(PUB,'publication_headline_grouped.csv'), index=False)
hl

,model,AUPRC ↑ (primary onset),Brier ↓,ECE ↓ (calibration),Coverage@90 → 0.90,PPR CRPS ↓ (proper),PPR RMSE ↓ (post-prefix),PPR RMSE worst-state ↓,N_liq log-MAE ↓ (censored-aware),N_liq log-MAE ↓ (liquefied-only),Early-warning rate ↑,CRR RMSE ↓ (DPI family),Physics violations ↓,AUROC ↑ (reference only)
0,EVT-NeuralSSM,"0.983 [0.907, 0.998]","0.052 [0.018, 0.125]","0.061 [0.020, 0.121]","0.788 [0.755, 0.881]",0.062,"0.120 [0.069, 0.137]",0.162,"0.281 [0.149, 0.381]",0.306,0.490,0.203,0.000,"0.981 [0.900, 0.997]"
1,DPI-Flow,"0.990 [0.947, 0.999]","0.044 [0.015, 0.103]","0.048 [0.015, 0.102]","0.844 [0.775, 0.897]",0.042,"0.087 [0.056, 0.109]",0.112,"0.268 [0.169, 0.328]",0.352,0.090,0.207,0.002,"0.988 [0.952, 0.998]"
2,DPI-EVT,"0.989 [0.930, 0.997]","0.047 [0.027, 0.137]","0.051 [0.026, 0.144]","0.851 [0.677, 0.884]",0.049,"0.098 [0.076, 0.149]",0.127,"0.252 [0.192, 0.314]",0.311,0.477,0.159,0.000,"0.986 [0.943, 0.995]"
3,Transformer,"0.964 [0.866, 0.999]","0.049 [0.014, 0.134]","0.071 [0.019, 0.135]","0.931 [0.886, 0.980]",0.026,"0.050 [0.022, 0.047]",0.096,"0.538 [0.416, 0.676]",0.672,0.470,—,0.089,"0.957 [0.821, 0.998]"
4,Neural Spline Flow,"0.985 [0.943, 0.998]","0.051 [0.024, 0.090]","0.068 [0.021, 0.079]","0.679 [0.668, 0.779]",0.061,"0.104 [0.060, 0.102]",0.130,"0.727 [0.593, 0.877]",0.878,0.446,—,0.799,"0.982 [0.954, 0.996]"
5,GRU,"0.945 [0.652, 0.975]","0.123 [0.085, 0.204]","0.164 [0.062, 0.206]","0.862 [0.774, 0.942]",0.067,"0.119 [0.064, 0.149]",0.187,"1.262 [1.169, 2.167]",1.470,0.289,—,0.514,"0.930 [0.737, 0.956]"
6,PINN,"0.951 [0.786, 0.993]","0.111 [0.048, 0.182]","0.150 [0.036, 0.167]","0.823 [0.682, 0.848]",0.085,"0.154 [0.069, 0.130]",0.240,"1.192 [0.654, 1.218]",1.480,0.290,—,0.000,"0.942 [0.758, 0.987]"
7,TCN,"0.935 [0.758, 0.982]","0.136 [0.067, 0.194]","0.149 [0.053, 0.193]","0.823 [0.699, 0.877]",0.084,"0.163 [0.067, 0.137]",0.266,"1.458 [0.817, 1.367]",1.745,0.310,—,0.677,"0.919 [0.752, 0.966]"
8,CatBoost,"0.996 [0.987, 1.000]","0.038 [0.004, 0.072]","0.051 [0.005, 0.077]",—,—,—,—,"0.214 [0.171, 0.263]",0.245,0.444,—,—,"0.993 [0.983, 0.999]"
